In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
import rasterstats
import matplotlib.pyplot as plt


In [ ]:
names = [
    'Broadleaf woodland',
    'Coniferous woodland',
    'Arable',
    'Improved grassland',
    'Semi-natural grassland',
    'Mountain, heath, bog',
    'Saltwater',
    'Freshwater',
    'Coastal',
    'Built-up areas and gardens'
]

values = range(1, 11)

landcover_names = dict(zip(values, names))

landcover_names

In [ ]:
with rio.open('project_data/LCM2015_Aggregate_100m.tif') as dataset:
    xmin, ymin, xmax, ymax = dataset.bounds
    crs = dataset.crs
    landcover = dataset.read(1)
    affine_tfm = dataset.transform

In [ ]:
wards = gpd.read_file('project_data/NI_wards.shp').to_crs(crs)

wards.head()

In [ ]:
# 4. 用老师的 zonal_stats 方法统计每个 ward 内的 land cover 像元数

ward_stats = rasterstats.zonal_stats(
    wards,
    landcover,
    affine=affine_tfm,
    categorical=True,
    category_map=landcover_names,
    nodata=0
)

print(type(ward_stats))
print(ward_stats[0])

In [ ]:
# 5. 把 zonal statistics 结果转成 DataFrame

ward_stats_df = pd.DataFrame(ward_stats)

ward_stats_df.head()

In [ ]:
wards_stats = wards.join(ward_stats_df)

wards_stats.head()

In [ ]:
# 7. 整理土地覆盖列，空值填 0

landcover_cols = list(landcover_names.values())

wards_stats[landcover_cols] = wards_stats[landcover_cols].fillna(0)

In [ ]:
# 8. 计算每个 ward 的总土地覆盖像元数

wards_stats["Total_pixels"] = wards_stats[landcover_cols].sum(axis=1)

wards_stats[["Ward", "Population", "Total_pixels"]].head()

In [ ]:
# 9. 计算每个 ward 中每种土地覆盖类型的比例

for col in landcover_cols:
    wards_stats[col + "_ratio"] = wards_stats[col] / wards_stats["Total_pixels"]

wards_stats.head()

In [ ]:
# 7. 整理土地覆盖列，空值填 0

landcover_cols = list(landcover_names.values())

wards_stats[landcover_cols] = wards_stats[landcover_cols].fillna(0)  # 

In [ ]:
# 8. 计算每个 ward 的总土地覆盖像元数

wards_stats["Total_pixels"] = wards_stats[landcover_cols].sum(axis=1)

wards_stats[["Ward", "Population", "Total_pixels"]].head()

In [ ]:
# 9. 计算每个 ward 中每种土地覆盖类型的比例

for col in landcover_cols:
    wards_stats[col + "_ratio"] = wards_stats[col] / wards_stats["Total_pixels"]

wards_stats.head()

In [ ]:
# 10. 重点：计算 Urban ratio

wards_stats["Urban_ratio"] = (
    wards_stats["Built-up areas and gardens"] / wards_stats["Total_pixels"]
)

wards_stats[["Ward", "Population", "Urban_ratio"]].head()

In [ ]:
# 11. 判断每个 ward 的主导土地覆盖类型

wards_stats["Dominant_landcover"] = wards_stats[landcover_cols].idxmax(axis=1)

wards_stats[["Ward", "Population", "Dominant_landcover"]].head()


In [ ]:
# 12. 人口 vs Urban ratio 相关性

correlation = wards_stats["Population"].corr(wards_stats["Urban_ratio"])

correlation

In [ ]:
# 13. 画散点图：Population vs Urban ratio

wards_stats.plot.scatter(
    x="Population",
    y="Urban_ratio",
    figsize=(7, 5)
)

plt.title("Population vs Urban Ratio")
plt.xlabel("Population")
plt.ylabel("Urban Ratio")
plt.show()


In [ ]:
# 14. 画 Urban ratio map

ax = wards_stats.plot(
    column="Urban_ratio",
    legend=True,
    figsize=(10, 8)
)

ax.set_title("Urban Ratio by Ward")
ax.set_axis_off()

plt.show()

In [ ]:
# 15. 画 Dominant land cover map

ax = wards_stats.plot(
    column="Dominant_landcover",
    legend=True,
    figsize=(10, 8)
)

ax.set_title("Dominant Land Cover by Ward")
ax.set_axis_off()

plt.show()


In [ ]:
# 16. 查看 Urban ratio 最高的 wards

wards_stats[
    ["Ward", "Population", "Urban_ratio", "Dominant_landcover"]
].sort_values(
    by="Urban_ratio",
    ascending=False
).head(10)


In [ ]:
# 17. 查看 Urban ratio 最低的 wards

wards_stats[
    ["Ward", "Population", "Urban_ratio", "Dominant_landcover"]
].sort_values(
    by="Urban_ratio",
    ascending=True
).head(10)

你可以把三者结合起来说：

👉 示例逻辑：

图1显示：大多数区域由 grassland / mountain 主导
图3显示：城市化集中在 Belfast 附近
图2验证：Urban ratio 最高的 ward 都在城市区域
📌 五、一句话总结

👉 三张图是同一分析的三种展示方式：

分类（Dominant）
数值（Urban ratio）
空间分布（地图）
🚀 最实用建议（帮你拿分）

如果只能交一个重点：

👉 交 图3（Urban ratio map） + 简单分析

如果想加分：

👉 再加 图1（Dominant map）

👉 图3是你发的最下面那张（带底图、可以点击的那张）

也就是这个👇特征：

有 OpenStreetMap 底图（像 Google 地图一样）
有缩放按钮
鼠标点一个区域会弹出：
Ward Code
Ward
Population
颜色是渐变的（蓝 → 绿 → 黄）

👉 这个就是：

Folium 做的 Urban ratio 交互地图（choropleth map）

In [ ]:
wards_stats["Urban_ratio"]

In [ ]:
import folium

m = folium.Map(location=[54.6, -6.7], zoom_start=8)

folium.Choropleth(
    geo_data=wards_stats,
    data=wards_stats,
    columns=["Ward", "Urban_ratio"],  # 👈 这里必须是 Urban_ratio
    key_on="feature.properties.Ward",
    fill_color="viridis",
    legend_name="Urban Ratio"
).add_to(m)

m

1. 城市化高度集中（最重要）

👉 从图上可以明显看到：

高 Urban ratio 区域非常少
且集中在：
Belfast 附近（东部）
少数城市点

👉 可以写：

Urban land cover is highly concentrated in a small number of wards, mainly around major urban centres such as Belfast.

2. 大多数区域是低城市化（农村为主）

👉 绝大部分区域是：

深紫色（Urban ratio 很低）

👉 说明：

北爱尔兰大多数是：
草地 / 农田 / 山地

👉 可以写：

Most wards exhibit very low urban ratios, indicating that the region is predominantly rural in character.

✅ 3. 城市呈“点状分布”（不是连续）

👉 你图里：

城市不是一大片
而是“小点”分布

👉 说明：

城市是离散的（town-based）

👉 可以写：

Built-up areas appear as small, discrete clusters rather than continuous urban regions.

✅ 4. 人口和城市化可能相关（结合你表）

你前面算过：

👉 Urban ratio 排名前几的 ward：

全是 built-up areas
人口也相对高

👉 可以写：

Wards with the highest urban ratios are typically associated with higher population values, suggesting a positive relationship between population and urban land cover.
The Urban Ratio map shows that built-up land is highly concentrated in a small number of wards, primarily located around major urban centres such as Belfast. Most wards display very low urban ratios, indicating that Northern Ireland is predominantly rural. The spatial pattern of urban land cover appears fragmented, with small clusters rather than continuous urban areas. This suggests that urban development is localised and not widespread across the region. Furthermore, wards with the highest urban ratios tend to have higher population values, indicating a positive relationship between population density and urban land use.


一、这整套代码在做什么（核心一句话）

👉 分析不同选区（wards）的土地利用结构，并研究它与人口的关系

🧠 二、分解成 3 个具体分析目标
✅ 1. 分析土地覆盖分布（基础）

通过：

zonal statistics

你得到：

👉 每个 ward 里：

森林多少
农田多少
城市多少

👉 回答的问题是：

不同区域的土地类型组成是怎样的？

✅ 2. 分析城市化程度（重点🔥）

通过：

Urban_ratio = built-up / total

👉 你得到：

哪些地方更城市化
哪些地方更农村

👉 回答的问题是：

哪些区域是城市？哪些是农村？

✅ 3. 分析人口 vs 土地利用（核心🔥）

你做了：

Urban_ratio vs Population
排序表

👉 回答的问题是：

人口多的地方，是否更城市化？

🗺️ 三、地图在干嘛？

你画了两种地图：

🟦 Dominant map

👉 回答：

每个区域主要是什么土地类型？

🟩 Urban ratio map（你最后那张）

👉 回答：

城市化在空间上怎么分布？

🧾 四、总结成一句标准作业答案（直接可用）

👉 你可以写：

This analysis uses zonal statistics to quantify land cover composition within each ward. It then calculates the proportion of built-up land (urban ratio) to assess levels of urbanisation. By combining these results with population data, the analysis explores the relationship between population distribution and land use patterns across Northern Ireland.

📌 五、再压缩成最简单一句

👉 你这套代码就是在研究：

👉 “人口分布如何影响（或反映）土地利用，特别是城市化程度”